# Embed the labeled crops that are missing from the classifier's training data

`crop_frames.py` made **1,614** labeled crops across FJI, SLB, TUV and VUT, but `image_labels.csv` /
`images_embeddings.json` only hold **1,118** of them: 496 FJI and VUT crops were never embedded (mostly sharks,
bony fish and cephalopods). This notebook embeds just those crops, the same way as the other 1,118, and gives
each one 2 augmented copies, as `augment_crops_colab.ipynb` did for the others.

1. finds every cropped row in each mission's `crops_manifest.csv` that has no embedding yet
2. checks that re-embedding 32 existing crops reproduces their stored embeddings (same model + preprocessing)
3. embeds the missing crops, then makes and embeds 2 augmented copies of each
4. saves 4 small files with **only the new rows** to `Matewos's Notebook/results/labeled_crops/missing_crops/`

**Before you start:** T4 GPU · a shortcut to **NatGeo Pristine Seas - FA26** in My Drive. The fine-tuned
`best.pt` (3.7 GB) is copied from `Matewos's Notebook/checkpoints/` if it's there, otherwise downloaded.
Nothing in the shared folder is changed.

**Afterwards (locally):** the new rows are merged into `checkpoints/labeled_crops/` and the classifier is retrained.

## 0. Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
import torch
print("GPU:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "NONE: works, but embedding is slow")

## 1. Settings

In [ ]:
from pathlib import Path
import glob, json, os, shutil, sys
import numpy as np
import pandas as pd

MY_DIR = Path("/content/drive/MyDrive/Matewos's Notebook")
BRANCH = "feature/cluster-classifier-suggestions"   # "main" once it's merged
COPIES_PER_CROP = 2      # same as the augmented crops already in checkpoints/labeled_crops/
SEED = 1                 # the other augmented crops used seed 0
OUT = MY_DIR / "results" / "labeled_crops" / "missing_crops"

hits = glob.glob("/content/drive/MyDrive/**/LABELED_CROPS", recursive=True)
assert hits, "LABELED_CROPS not found: add a shortcut to 'NatGeo Pristine Seas - FA26' in My Drive"
LC = Path(hits[0])
print("LABELED_CROPS:", LC)

## 2. Code, model weights and the labeled crops

In [ ]:
!rm -rf /content/nat-geo-fa26
!git clone -q --depth 1 -b {BRANCH} https://github.com/LaurenMaiH/nat-geo-fa26.git /content/nat-geo-fa26
!pip -q install -r /content/nat-geo-fa26/requirements.txt
REPO = Path("/content/nat-geo-fa26")
CKPT_DRIVE = MY_DIR / "checkpoints" / "best.pt"
if CKPT_DRIVE.exists():
    shutil.copy(CKPT_DRIVE, REPO / "checkpoints" / "best.pt")
    print("fine-tuned best.pt copied from Drive")

labels = pd.read_csv(REPO / "checkpoints" / "labeled_crops" / "image_labels.csv")
stored = json.loads((REPO / "checkpoints" / "labeled_crops" / "images_embeddings.json").read_text())
print(f"{len(labels)} labeled crops already embedded")

In [ ]:
# Each mission's crops_manifest.csv says where its crop images are (paths were written on a
# teammate's laptop, so each crop is found by the tail of its path inside the mission folder).
def resolve(p, root):
    parts = [x for x in str(p).replace("\\", "/").split("/") if x]
    for n in range(len(parts), 0, -1):
        cand = root / "/".join(parts[-n:])
        if cand.exists():
            return str(cand)
    return None

parts = []
for f in sorted(LC.glob("*/crops_manifest.csv")):
    m = pd.read_csv(f, low_memory=False)
    m = m[m["crop_status"].astype(str) == "cropped"].copy()
    m["mission"] = f.parent.name.split("_")[0]
    m["crop_path"] = [resolve(p, f.parent) for p in m["crop_path"]]
    parts.append(m)
man = pd.concat(parts, ignore_index=True).drop_duplicates(["deployment", "row"])
man["key"] = man["deployment"].astype(str) + "__row" + man["row"].astype(int).astype(str)
print(f"{len(man)} cropped rows in the manifests; {man['crop_path'].isna().sum()} crop images not found")

path_of = dict(zip(man["key"], man["crop_path"]))
assert labels["key"].isin(path_of).mean() > 0.99, "existing labels don't line up with the manifests"

missing = man[~man["key"].isin(set(labels["key"]))].dropna(subset=["crop_path"])
print(f"{len(missing)} crops to embed:", missing["mission"].value_counts().to_dict())
print(missing["class"].value_counts().to_string())

# same columns as image_labels.csv
new_labels = missing.assign(n_images=1)[list(labels.columns)].reset_index(drop=True)

## 3. Load the fine-tuned BioCLIP, and check it reproduces the stored embeddings
Similarity should be ~1.00. If it isn't, the new crops would be embedded differently from the old ones, so the notebook stops.

In [ ]:
os.chdir(REPO)
sys.path.insert(0, str(REPO))
import cv2
from PIL import Image
import overnight_pipeline_optimized as pipe
pipe.INFERENCE_DEVICE = pipe.resolve_device(None)
embed_info = pipe.load_embedding_model()
assert embed_info[0] == "finetuned", f"BioCLIP weights: {embed_info[0]}; need the fine-tuned best.pt"

def embed(pil_images, batch=64):
    out = []
    for i in range(0, len(pil_images), batch):
        bgr = [cv2.cvtColor(np.asarray(im.convert("RGB")), cv2.COLOR_RGB2BGR) for im in pil_images[i:i + batch]]
        out.append(pipe.get_embeddings_batch(embed_info, bgr))
    return np.concatenate(out)

check = labels.sample(32, random_state=0)
mine = embed([Image.open(path_of[k]) for k in check["key"]])
theirs = np.asarray([stored[k] for k in check["key"]], dtype=np.float32)
theirs /= np.linalg.norm(theirs, axis=1, keepdims=True)
sim = (mine * theirs).sum(axis=1)
print(f"similarity to stored embeddings: min {sim.min():.4f}, median {np.median(sim):.4f}")
assert sim.min() > 0.98, "re-embedded crops don't match the stored embeddings: different model or preprocessing"

## 4. Embed the missing crops

In [ ]:
vecs = embed([Image.open(p) for p in missing["crop_path"]])
new_emb = {k: [float(x) for x in v] for k, v in zip(new_labels["key"], vecs)}
print(f"{len(new_emb)} crops embedded")

## 5. Augmented copies of the new crops
The same augmentation as `augment_crops_colab.ipynb`: rotation ±15°, shift + zoom, horizontal flip, lighting, blur and noise.

In [ ]:
from PIL import Image, ImageEnhance, ImageFilter, ImageOps

def augment(img, rng):
    img = img.convert("RGB")
    w, h = img.size
    a = np.asarray(img)
    pad = int(0.25 * max(w, h))
    big = Image.fromarray(np.pad(a, ((pad, pad), (pad, pad), (0, 0)), mode="reflect"))
    angle = rng.uniform(-15, 15)
    big = big.rotate(angle, resample=Image.BICUBIC)
    s = rng.uniform(0.75, 1.0)
    dx, dy = rng.uniform(-0.1, 0.1) * w, rng.uniform(-0.1, 0.1) * h
    cx, cy = pad + w / 2 + dx, pad + h / 2 + dy
    out = big.crop((cx - s * w / 2, cy - s * h / 2, cx + s * w / 2, cy + s * h / 2)).resize((w, h), Image.BICUBIC)
    flip = rng.random() < 0.5
    if flip:
        out = ImageOps.mirror(out)
    b, c, sat = rng.uniform(0.8, 1.2), rng.uniform(0.8, 1.2), rng.uniform(0.8, 1.1)
    out = ImageEnhance.Color(ImageEnhance.Contrast(ImageEnhance.Brightness(out).enhance(b)).enhance(c)).enhance(sat)
    blur = rng.uniform(0.5, 1.5) if rng.random() < 0.3 else 0
    if blur:
        out = out.filter(ImageFilter.GaussianBlur(blur))
    noise = rng.uniform(3, 10) if rng.random() < 0.3 else 0
    if noise:
        arr = np.asarray(out).astype(np.float32) + rng.normal(0, noise, (h, w, 3))
        out = Image.fromarray(np.clip(arr, 0, 255).astype(np.uint8))
    params = f"rot={angle:.0f} zoom={s:.2f} shift=({dx / w:.2f},{dy / h:.2f}) flip={int(flip)} " \
             f"bright={b:.2f} contrast={c:.2f} sat={sat:.2f} blur={blur:.1f} noise={noise:.0f}"
    return out, params

In [ ]:
from tqdm.auto import tqdm
rng = np.random.default_rng(SEED)
rows, aug_emb = [], {}
pending_imgs, pending_keys = [], []

def flush():
    if pending_imgs:
        for k, v in zip(pending_keys, embed(pending_imgs)):
            aug_emb[k] = [round(float(x), 6) for x in v]
        pending_imgs.clear(); pending_keys.clear()

# Plain dicts, not itertuples(): a column named "class" (a Python keyword) gets renamed there
for r in tqdm(new_labels.to_dict("records")):
    img = Image.open(path_of[r["key"]]).convert("RGB")
    for i in range(COPIES_PER_CROP):
        im, params = augment(img, rng)
        k = f"{r['key']}__aug{i}"
        rows.append({**r, "key": k, "augmented_from": r["key"], "augmentation": params})
        pending_imgs.append(im); pending_keys.append(k)
        if len(pending_imgs) >= 256:
            flush()
flush()
new_aug_labels = pd.DataFrame(rows)
print(f"{len(new_aug_labels)} augmented crops of {new_aug_labels['augmented_from'].nunique()} originals")

## 6. Save (only the new rows)

In [ ]:
OUT.mkdir(parents=True, exist_ok=True)
new_labels.to_csv(OUT / "new_image_labels.csv", index=False)
(OUT / "new_images_embeddings.json").write_text(json.dumps(new_emb))
new_aug_labels.to_csv(OUT / "new_augmented_labels.csv", index=False)
(OUT / "new_augmented_embeddings.json").write_text(json.dumps(aug_emb))
print("saved to", OUT)
for f in sorted(OUT.iterdir()):
    print(f"  {f.name}  {f.stat().st_size / 1e6:.1f} MB")